In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import sqlite3

### Setup

In [ ]:
CONFLICT_DATA_PATH = "C:/Users/liivas/Downloads/Töö/estnltk_projekt_2025/morphology_conflicts/data/"
TRANSACTION_DATA_PATH = "C:/Users/liivas/Downloads/Töö/estnltk_projekt_2025/transaktsioonid/source_data/"
CONFLICTS = "conflict_results.db"
TRANSACTIONS = "v33_koondkorpus_sentences_verb_pattern_obl_20241002-130310.db"

### Functions for calculating error rate and impact

In [13]:
def calculate_error_rate(deprel: str, 
                         conflict_db_path: str, 
                         transaction_db_path: str, 
                         conflict_table_name: str, 
                         tr_row_table_name: str) -> float:
    
    con = sqlite3.connect(conflict_db_path)
    cur = con.cursor()
    cur.execute(f'ATTACH DATABASE "{transaction_db_path}" AS tr')

    cur.execute("""
    SELECT
        count(*)
    FROM
        {conflict_table_name}
    """.format(conflict_table_name=conflict_table_name))
    
    n_mistakes = cur.fetchone()[0]

    cur.execute("""
    SELECT
        count(DISTINCT tr_row.head_id)
    FROM
        tr.{tr_row_table_name} AS tr_row
    WHERE
        tr_row.deprel = '{deprel}'
    """.format(tr_row_table_name=tr_row_table_name, deprel=deprel))
    
    n_all_instances = cur.fetchone()[0]

    con.close()

    error_rate = n_mistakes / n_all_instances

    return error_rate * 100000


def calculate_impact(conflict_db_path: str, 
                     transaction_db_path: str, 
                     conflict_table_name: str, 
                     tr_head_table_name: str) -> float:
    
    con = sqlite3.connect(conflict_db_path)
    cur = con.cursor()
    cur.execute(f'ATTACH DATABASE "{transaction_db_path}" AS tr')

    cur.execute("""
    SELECT
        count(*)
    FROM
        {conflict_table_name}
    """.format(conflict_table_name=conflict_table_name))
    
    n_mistakes = cur.fetchone()[0]

    cur.execute("""
    SELECT
        count(*)
    FROM
        {tr_head_table_name}
    """.format(tr_head_table_name=tr_head_table_name))
    
    n_all_tr_heads = cur.fetchone()[0]

    con.close()

    impact = n_mistakes / n_all_tr_heads

    return impact * 100000

### I Error rate and impact of **nsubj ^(nom|part)**

In [ ]:
nsubj_error_rate = calculate_error_rate("nsubj", 
                                        f"{CONFLICT_DATA_PATH}{CONFLICTS}", 
                                        f"{TRANSACTION_DATA_PATH}{TRANSACTIONS}", 
                                        "nsubj_not_nom_part", 
                                        "transaction_row")

nsubj_impact = calculate_impact(f"{CONFLICT_DATA_PATH}{CONFLICTS}", 
                                f"{TRANSACTION_DATA_PATH}{TRANSACTIONS}", 
                                "nsubj_not_nom_part", 
                                "transaction_head")

In [ ]:
# 1:100000
print(nsubj_error_rate)
print(nsubj_impact)

68.20027025286126
32.03232342360409


### II Error rate and impact of **obj ^(nom|gen|part)**

In [18]:
obj_error_rate = calculate_error_rate("obj", 
                                        f"{CONFLICT_DATA_PATH}{CONFLICTS}", 
                                        f"{TRANSACTION_DATA_PATH}{TRANSACTIONS}", 
                                        "obj_not_nom_gen_part", 
                                        "transaction_row")

obj_impact = calculate_impact(f"{CONFLICT_DATA_PATH}{CONFLICTS}", 
                                f"{TRANSACTION_DATA_PATH}{TRANSACTIONS}", 
                                "obj_not_nom_gen_part", 
                                "transaction_head")

In [19]:
# 1:100000
print(obj_error_rate)
print(obj_impact)

892.952636053939
193.46725448778338


### III Error rate and impact of **advcl gen**

In [20]:
advcl_error_rate = calculate_error_rate("advcl", 
                                        f"{CONFLICT_DATA_PATH}{CONFLICTS}", 
                                        f"{TRANSACTION_DATA_PATH}{TRANSACTIONS}", 
                                        "advcl_gen", 
                                        "transaction_row")

advcl_impact = calculate_impact(f"{CONFLICT_DATA_PATH}{CONFLICTS}", 
                                f"{TRANSACTION_DATA_PATH}{TRANSACTIONS}", 
                                "advcl_gen", 
                                "transaction_head")

In [21]:
# 1:100000
print(advcl_error_rate)
print(advcl_impact)

24.378941430096912
1.609096475041451
